# DAG最长路径算法详解

## 基本概念

在**有向无环图(DAG)**中寻找最长路径是一个经典问题，与最短路径问题不同，最长路径问题在一般图中是NP难的，但在DAG中可以在多项式时间内解决。

### 关键特性
- **无环性**：DAG没有循环，确保不会出现无限长的路径
- **可拓扑排序**：DAG可以进行拓扑排序，为动态规划提供基础
- **负权边处理**：算法可以处理包含负权边的情况

## 算法原理

### 核心思想
利用DAG的拓扑排序特性，按拓扑顺序进行动态规划计算。

### 算法步骤

1. **拓扑排序**：对DAG进行拓扑排序
2. **初始化**：设源点距离为0，其他点为负无穷
3. **动态规划**：按拓扑顺序处理每个顶点，更新其邻居的距离
4. **路径重构**：通过前驱节点重构最长路径

## Python实现

### 基础版本

In [ ]:
from collections import defaultdict, deque

class DAGLongestPath:
    def __init__(self, vertices):
        self.V = vertices
        self.graph = defaultdict(list)
        self.indegree = [0] * vertices
    
    def add_edge(self, u, v, weight):
        """添加有向边 u -> v，权重为weight"""
        self.graph[u].append((v, weight))
        self.indegree[v] += 1
    
    def topological_sort(self):
        """拓扑排序"""
        indegree = self.indegree.copy()
        queue = deque()
        topo_order = []
        
        # 入度为0的节点入队
        for i in range(self.V):
            if indegree[i] == 0:
                queue.append(i)
        
        while queue:
            u = queue.popleft()
            topo_order.append(u)
            
            for v, weight in self.graph[u]:
                indegree[v] -= 1
                if indegree[v] == 0:
                    queue.append(v)
        
        return topo_order
    
    def longest_path(self, start):
        """
        计算从start开始的最长路径
        
        返回:
        dist: 最长距离数组
        prev: 前驱节点数组
        """
        # 拓扑排序
        topo_order = self.topological_sort()
        
        # 初始化距离和前驱数组
        dist = [-float('inf')] * self.V
        prev = [-1] * self.V
        dist[start] = 0
        
        # 按拓扑顺序处理每个节点
        for u in topo_order:
            if dist[u] != -float('inf'):
                for v, weight in self.graph[u]:
                    if dist[v] < dist[u] + weight:
                        dist[v] = dist[u] + weight
                        prev[v] = u
        
        return dist, prev
    
    def get_longest_path(self, start, end, dist, prev):
        """重构从start到end的最长路径"""
        if dist[end] == -float('inf'):
            return None, -float('inf')
        
        path = []
        current = end
        
        while current != -1:
            path.append(current)
            current = prev[current]
        
        path.reverse()
        return path, dist[end]

# 使用示例
dag = DAGLongestPath(6)

# 添加边（可以包含负权重）
dag.add_edge(0, 1, 5)
dag.add_edge(0, 2, 3)
dag.add_edge(1, 3, 6)
dag.add_edge(1, 2, 2)
dag.add_edge(2, 4, 4)
dag.add_edge(2, 5, 2)
dag.add_edge(2, 3, 7)
dag.add_edge(3, 5, 1)
dag.add_edge(3, 4, -1)  # 负权边
dag.add_edge(4, 5, -2)  # 负权边

dist, prev = dag.longest_path(0)
path, length = dag.get_longest_path(0, 5, dist, prev)

print(f"从0到5的最长路径长度: {length}")
print(f"路径: {path}")


### 优化版本（支持负权边和路径重构）

In [ ]:
from collections import defaultdict, deque
import heapq

class AdvancedDAGLongestPath:
    def __init__(self):
        self.graph = defaultdict(list)
        self.nodes = set()
    
    def add_edge(self, u, v, weight):
        """添加有向边"""
        self.graph[u].append((v, weight))
        self.nodes.add(u)
        self.nodes.add(v)
    
    def get_indegree(self):
        """计算所有节点的入度"""
        indegree = defaultdict(int)
        for u in self.graph:
            for v, w in self.graph[u]:
                indegree[v] += 1
            if u not in indegree:
                indegree[u] = 0
        return indegree
    
    def topological_sort(self):
        """拓扑排序"""
        indegree = self.get_indegree()
        queue = deque([node for node in self.nodes if indegree[node] == 0])
        topo_order = []
        
        while queue:
            u = queue.popleft()
            topo_order.append(u)
            
            for v, weight in self.graph[u]:
                indegree[v] -= 1
                if indegree[v] == 0:
                    queue.append(v)
        
        return topo_order
    
    def find_longest_paths_from_source(self, start):
        """
        找到从源点start到所有其他节点的最长路径
        
        返回:
        dist: 距离字典
        prev: 前驱节点字典
        """
        topo_order = self.topological_sort()
        
        # 初始化
        dist = {node: -float('inf') for node in self.nodes}
        prev = {node: None for node in self.nodes}
        dist[start] = 0
        
        # 动态规划
        for u in topo_order:
            if dist[u] != -float('inf'):
                for v, weight in self.graph[u]:
                    if dist[v] < dist[u] + weight:
                        dist[v] = dist[u] + weight
                        prev[v] = u
        
        return dist, prev
    
    def find_longest_path_in_dag(self):
        """
        找到整个DAG中的最长路径（不指定起点终点）
        
        返回:
        longest_path: 最长路径
        max_length: 最长路径长度
        """
        max_length = -float('inf')
        best_path = []
        best_start = None
        
        # 对每个可能的起点计算最长路径
        for start in self.nodes:
            dist, prev = self.find_longest_paths_from_source(start)
            
            # 找到从这个起点出发的最长路径
            for end in self.nodes:
                if dist[end] > max_length:
                    max_length = dist[end]
                    best_start = start
                    best_end = end
        
        # 重构路径
        if best_start is not None:
            dist, prev = self.find_longest_paths_from_source(best_start)
            best_path = self._reconstruct_path(prev, best_start, best_end)
        
        return best_path, max_length
    
    def _reconstruct_path(self, prev, start, end):
        """重构路径"""
        path = []
        current = end
        
        while current is not None:
            path.append(current)
            current = prev[current]
        
        path.reverse()
        return path if path[0] == start else []
    
    def find_critical_path(self):
        """
        关键路径算法 - 最长路径在项目管理中的应用
        假设图表示项目的活动网络
        """
        # 添加虚拟起点和终点
        all_nodes = self.nodes.copy()
        
        # 找到所有起点（入度为0）和终点（出度为0）
        indegree = self.get_indegree()
        starts = [node for node in all_nodes if indegree[node] == 0]
        ends = [node for node in all_nodes if not self.graph[node]]
        
        # 添加虚拟超级起点和超级终点
        super_start = "start"
        super_end = "end"
        
        # 临时添加边
        for start in starts:
            self.add_edge(super_start, start, 0)
        for end in ends:
            self.add_edge(end, super_end, 0)
        
        # 计算最长路径
        dist, prev = self.find_longest_paths_from_source(super_start)
        critical_path = self._reconstruct_path(prev, super_start, super_end)
        
        # 移除虚拟节点
        critical_path = [node for node in critical_path if node != super_start and node != super_end]
        
        return critical_path, dist[super_end]

# 使用示例
dag = AdvancedDAGLongestPath()

# 构建一个项目网络图（活动节点，边权重表示持续时间）
dag.add_edge("A", "B", 3)
dag.add_edge("A", "C", 2)
dag.add_edge("B", "D", 4)
dag.add_edge("C", "D", 1)
dag.add_edge("C", "E", 3)
dag.add_edge("D", "F", 2)
dag.add_edge("E", "F", 2)
dag.add_edge("F", "G", 1)

print("=== DAG最长路径算法演示 ===")

# 1. 从特定起点计算
print("\n1. 从A出发的最长路径:")
dist, prev = dag.find_longest_paths_from_source("A")
for node in dag.nodes:
    path = dag._reconstruct_path(prev, "A", node)
    if dist[node] != -float('inf'):
        print(f"A -> {node}: 长度={dist[node]}, 路径={' -> '.join(path)}")

# 2. 整个DAG的最长路径
print("\n2. 整个DAG的最长路径:")
longest_path, max_length = dag.find_longest_path_in_dag()
print(f"最长路径: {' -> '.join(longest_path)}")
print(f"路径长度: {max_length}")

# 3. 关键路径分析
print("\n3. 关键路径分析:")
critical_path, total_duration = dag.find_critical_path()
print(f"关键路径: {' -> '.join(critical_path)}")
print(f"项目总工期: {total_duration}")


### 处理负权边的特殊案例

In [ ]:
def longest_path_with_negative_weights():
    """演示包含负权边的DAG最长路径"""
    dag = AdvancedDAGLongestPath()
    
    # 添加包含负权重的边
    dag.add_edge("A", "B", 5)
    dag.add_edge("A", "C", 3)
    dag.add_edge("B", "D", -2)  # 负权边
    dag.add_edge("C", "D", 4)
    dag.add_edge("C", "E", -1)  # 负权边
    dag.add_edge("D", "F", 6)
    dag.add_edge("E", "F", 2)
    dag.add_edge("F", "G", -3)  # 负权边
    
    print("=== 包含负权边的DAG最长路径 ===")
    dist, prev = dag.find_longest_paths_from_source("A")
    
    for node in sorted(dag.nodes):
        path = dag._reconstruct_path(prev, "A", node)
        if dist[node] != -float('inf'):
            print(f"A -> {node}: 长度={dist[node]:.1f}, 路径={' -> '.join(path)}")

# 运行示例
longest_path_with_negative_weights()


## 算法复杂度分析

- **时间复杂度**：O(V + E)
  - 拓扑排序：O(V + E)
  - 动态规划：O(V + E)
- **空间复杂度**：O(V)

## 应用场景

### 1. 关键路径法(CPM)
- **项目管理**：确定项目完成的最短时间
- **任务调度**：识别关键任务（延迟会影响总工期）

### 2. 系统依赖分析
- **编译顺序**：确定源代码文件的编译顺序
- **课程安排**：确定课程的学习顺序

### 3. 数据流分析
- **计算图优化**：在神经网络中寻找计算路径
- **流水线调度**：优化处理流程

## 与最短路径算法的对比

| 特性       | DAG最长路径        | Dijkstra最短路径 |
| ---------- | ------------------ | ---------------- |
| 图类型     | 有向无环图         | 带权图（无负环） |
| 权重限制   | 可处理负权边       | 不能处理负权边   |
| 时间复杂度 | O(V+E)             | O((V+E)logV)     |
| 应用场景   | 关键路径、依赖分析 | 路由、导航       |

## 注意事项

1. **负权环检测**：虽然DAG无环，但输入验证仍很重要
2. ** disconnected图**：需要处理不连通的情况
3. **数值稳定性**：使用足够小的负无穷值
4. **路径存在性**：检查路径是否实际存在

DAG最长路径算法因其高效性和处理负权边的能力，在项目规划、系统分析和运筹学等领域有着广泛应用。